# Basketball Tracker – YOLOv8

Detects and tracks a basketball in video clips using a YOLOv8n model fine-tuned on a Roboflow dataset.

**Pipeline:**
1. Install dependencies
2. Mount Google Drive
3. Download dataset from Roboflow
4. Train YOLOv8
5. Evaluate (mAP, precision, recall)
6. Run inference on test images
7. Track ball in a video with a motion trail
8. Save model weights to Drive

> **Note:** Run in the Colab browser (colab.research.google.com), not via a local VSCode kernel.

## 1. Install dependencies

In [ ]:
!pip install ultralytics roboflow --quiet

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/assignment_4'
os.makedirs(DRIVE_PATH, exist_ok=True)
print("Drive mounted. Output folder:", DRIVE_PATH)

## 3. Download dataset from Roboflow

In [ ]:
from roboflow import Roboflow
import os, shutil, random, yaml

ROBOFLOW_API_KEY = "fksw0LqQsvkjaDIQYsex"
WORKSPACE        = "tomass-workspace-tk20e"
PROJECT          = "basketball-ball-detection-hpri4-ay80u"
VERSION          = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

DATA_DIR = dataset.location
print("Dataset location:", DATA_DIR)
print("Contents:", os.listdir(DATA_DIR))

In [ ]:
# Dataset has no train split — create one from valid (80/20)
train_img = os.path.join(DATA_DIR, 'train', 'images')
train_lbl = os.path.join(DATA_DIR, 'train', 'labels')

if not os.path.exists(train_img):
    print("No train split found — creating one from valid (80/20)")
    val_img = os.path.join(DATA_DIR, 'valid', 'images')
    val_lbl = os.path.join(DATA_DIR, 'valid', 'labels')

    images = sorted(os.listdir(val_img))
    random.seed(42)
    random.shuffle(images)
    split = int(len(images) * 0.8)
    train_imgs = images[:split]
    new_val_imgs = images[split:]

    os.makedirs(train_img, exist_ok=True)
    os.makedirs(train_lbl, exist_ok=True)

    for fname in train_imgs:
        shutil.copy(os.path.join(val_img, fname), os.path.join(train_img, fname))
        lbl = fname.rsplit('.', 1)[0] + '.txt'
        src = os.path.join(val_lbl, lbl)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(train_lbl, lbl))

    for fname in train_imgs:
        os.remove(os.path.join(val_img, fname))
        lbl = fname.rsplit('.', 1)[0] + '.txt'
        lbl_path = os.path.join(val_lbl, lbl)
        if os.path.exists(lbl_path):
            os.remove(lbl_path)

    yaml_path = os.path.join(DATA_DIR, 'data.yaml')
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    cfg['train'] = train_img
    cfg['val']   = val_img
    cfg['test']  = os.path.join(DATA_DIR, 'test', 'images') if os.path.exists(os.path.join(DATA_DIR, 'test', 'images')) else val_img
    with open(yaml_path, 'w') as f:
        yaml.dump(cfg, f)

    print(f"Train: {len(train_imgs)} images, Val: {len(new_val_imgs)} images")
else:
    print("Train split exists:", len(os.listdir(train_img)), "images")

## 4. Train YOLOv8

We use `yolov8n` (nano), the smallest and fastest variant in the YOLOv8 family. Well suited for a focused single-class task like basketball detection.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=os.path.join(DATA_DIR, 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    name='basketball_detector',
    patience=10,
    device=0
)

## 5. Evaluate the model

Key metrics:
- **mAP50** – mean average precision at IoU threshold 0.5, the primary object detection metric
- **Precision** – of all predicted bounding boxes, how many were correct?
- **Recall** – of all actual balls in the dataset, how many did the model find?

In [ ]:
import glob as gl

# Locate the best model dynamically regardless of run suffix (-2, -3, etc.)
candidates = sorted(gl.glob('runs/detect/basketball_detector*/weights/best.pt'))
best_model_path = candidates[-1]
print("Loading:", best_model_path)

model = YOLO(best_model_path)

metrics = model.val(data=os.path.join(DATA_DIR, 'data.yaml'))

print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

In [ ]:
# Training curves (loss + mAP per epoch)
run_dir = best_model_path.replace('/weights/best.pt', '')

from IPython.display import Image
Image(os.path.join(run_dir, 'results.png'))

In [ ]:
Image(os.path.join(run_dir, 'confusion_matrix.png'))

In [ ]:
Image(os.path.join(run_dir, 'PR_curve.png'))

## 6. Inference on test images

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

test_images = glob.glob(f"{DATA_DIR}/test/images/*.jpg")[:6]
if not test_images:
    test_images = glob.glob(f"{DATA_DIR}/valid/images/*.jpg")[:6]

pred_results = model.predict(source=test_images, conf=0.25, save=True)

# Locate saved predictions dynamically
predict_dirs = sorted(glob.glob('runs/detect/predict*'))
predict_dir = predict_dirs[-1]
pred_images = sorted(glob.glob(f'{predict_dir}/*.jpg'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, pred_images):
    img = mpimg.imread(img_path)
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.suptitle('Detections on test images', fontsize=14, y=1.02)
plt.show()

## 7. Track the ball in a video with a motion trail

Upload a basketball video from your computer.

In [ ]:
from google.colab import files

uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print("Video:", VIDEO_PATH)

In [ ]:
import cv2
import numpy as np
from collections import deque

OUTPUT_PATH = '/content/basketball_tracked.avi'
TRAIL_LENGTH = 30
CONF_THRESHOLD = 0.3

trail = deque(maxlen=TRAIL_LENGTH)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video: {w}x{h} @ {fps:.1f} fps")

out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*'XVID'), fps, (w, h))

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    res = model.predict(frame, conf=CONF_THRESHOLD, verbose=False)
    boxes = res[0].boxes

    center = None
    if boxes is not None and len(boxes) > 0:
        best = boxes[boxes.conf.argmax()]
        x1, y1, x2, y2 = best.xyxy[0].cpu().numpy().astype(int)
        center = ((x1 + x2) // 2, (y1 + y2) // 2)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 200, 255), 2)
        cv2.putText(frame, f"ball {float(best.conf):.2f}", (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2)

    trail.append(center)

    valid = [(p, i) for i, p in enumerate(trail) if p is not None]
    for idx, (pos, i) in enumerate(valid[:-1]):
        next_pos = valid[idx + 1][0]
        alpha = (i + 1) / TRAIL_LENGTH
        thickness = max(1, int(alpha * 6))
        color = (int(50 * alpha), int(100 + 155 * alpha), 255)
        cv2.line(frame, pos, next_pos, color, thickness)

    out.write(frame)
    frame_count += 1

cap.release()
out.release()

size = os.path.getsize(OUTPUT_PATH) if os.path.exists(OUTPUT_PATH) else 0
print(f"Done! {frame_count} frames. File size: {size / 1024 / 1024:.1f} MB")

In [ ]:
# Convert to mp4 for in-browser preview
!ffmpeg -i /content/basketball_tracked.avi -vcodec libx264 /content/basketball_tracked.mp4 -y -loglevel quiet

from IPython.display import HTML
from base64 import b64encode

mp4 = open('/content/basketball_tracked.mp4', 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
HTML(f'<video width=640 controls><source src="{data_url}" type="video/mp4"></video>')

## 8. Save model and results to Drive

In [ ]:
import shutil

shutil.copy(best_model_path, f"{DRIVE_PATH}/best.pt")
shutil.copy('/content/basketball_tracked.mp4', f"{DRIVE_PATH}/basketball_tracked.mp4")
shutil.copy(os.path.join(run_dir, 'results.png'), f"{DRIVE_PATH}/results.png")
shutil.copy(os.path.join(run_dir, 'confusion_matrix.png'), f"{DRIVE_PATH}/confusion_matrix.png")
shutil.copy(os.path.join(run_dir, 'PR_curve.png'), f"{DRIVE_PATH}/PR_curve.png")

print("Saved to Drive:")
for f in os.listdir(DRIVE_PATH):
    print(" ", f)

## Summary

| Metric | Value |
|--------|-------|
| Model  | YOLOv8n (fine-tuned) |
| Dataset | Basketball-ball-detection – Roboflow (~4300 images, 1 class) |
| Train / Val split | 3154 / 789 images |
| Epochs | 50 |
| mAP50  | 0.803 |
| Precision | 0.842 |
| Recall | 0.705 |
| mAP50-95 | 0.553 |
| Inference speed | 0.5ms per frame (NVIDIA L4) |

**What worked:** The model is pretty good at finding the ball when it's clearly visible. The motion trail makes the result look cool and easy to follow.

**What didn't work:** When the ball moves really fast or a player steps in front of it, the model loses track. That was hard to fix just by adjusting settings.

**What I learned:** I didn't expect transfer learning to work this well on a small dataset. The model picked it up fast because it already knew what shapes and edges look like from being trained on millions of other images. The biggest surprise was that the dataset didn't have a train split at all, so I had to create one myself by splitting the data manually in code. I also learned that the confidence setting matters a lot. If it's too low the model starts detecting things that aren't the ball, if it's too high it misses the ball when things get messy. The thing that took the most time was actually building the app around the model. Training took 18 minutes but getting everything to work together as a real product took way longer than that.